# TP -- Dérivées, gradients... et le Mont-Blanc

Dans ce TP, nous allons manipuler les notions de **dérivée** et de **gradient** vues en cours, non pas sur des fonctions inventées, mais sur une **vraie carte d'altitude du massif du Mont-Blanc**.

L'intérêt est que tout le vocabulaire du cours est en fait du vocabulaire de montagne:

| Cours | Montagne |
|---|---|
| la dérivée $f'(x)$ | la **pente** (les $8\%$ des panneaux routiers) |
| le gradient $\nabla f(\vec{x})$ | la ligne de **plus grande pente** |
| sa norme $\| \nabla f(\vec{x}) \|$ | la **raideur** de la pente |
| un **maximum local** | un **sommet** |
| un **minimum local** | une **cuvette**, un lac |
| un **point selle** | un **col** |
| la direction $-\nabla f(\vec{x})$ | le chemin que suit une **goutte d'eau** |

À la fin du TP, votre code aura retrouvé tout seul les sommets du massif, et fait ruisseler des gouttes d'eau jusque dans la vallée de Chamonix.

## Librairies

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import maximum_filter, gaussian_filter

plt.rcParams["figure.figsize"] = (9, 6)

## 0. Les données: une carte d'altitude réelle

Le fichier `data/mont_blanc.npz` contient un **modèle numérique de terrain** (MNT) issu de la mission spatiale **SRTM** de la NASA (données publiques, résolution 1 seconde d'arc, soit environ 30 m).

C'est un tableau `numpy` de nombres: `altitude[i, j]` est l'altitude en mètres du point situé sur la ligne `i` et la colonne `j`.

Autrement dit, c'est exactement une fonction
$$
f : \mathbb{R}^2 \longrightarrow \mathbb{R}, \qquad f(x, y) = \text{altitude au point } (x,y),
$$
mais connue seulement en un nombre fini de points (une **grille**).

Attention à deux choses:

- la ligne `0` est au **sud** et la ligne du haut au **nord**: l'axe $y$ est donc orienté vers le nord, comme en maths;
- un pixel ne fait **pas** la même largeur que sa hauteur: `dx` mètres horizontalement (est-ouest) et `dy` mètres verticalement (nord-sud). Nous en aurons besoin pour calculer des pentes correctes.

In [ ]:
data = np.load("data/mont_blanc.npz")

Z   = data["altitude"].astype(float)   # altitudes en mètres
lat = data["lat"]                      # latitude de chaque ligne
lon = data["lon"]                      # longitude de chaque colonne
dx  = float(data["dx"])                # largeur d'un pixel, en mètres (est-ouest)
dy  = float(data["dy"])                # hauteur d'un pixel, en mètres (nord-sud)

print("forme du tableau :", Z.shape)
print("altitude min : %.0f m   max : %.0f m" % (Z.min(), Z.max()))
print("taille d'un pixel : dx = %.2f m,  dy = %.2f m" % (dx, dy))
print("source :", data["source"])

In [ ]:
# largeur et hauteur de la zone, en kilomètres
W = (Z.shape[1] - 1) * dx / 1000
H = (Z.shape[0] - 1) * dy / 1000
extent = [0, W, 0, H]     # pour que matplotlib affiche des kilomètres

plt.imshow(Z, origin="lower", extent=extent, cmap="terrain")
plt.colorbar(label="altitude (m)")
plt.xlabel("x (km, vers l'est)")
plt.ylabel("y (km, vers le nord)")
plt.title("Massif du Mont-Blanc -- carte d'altitude")
plt.show()

<div class="alert alert-block alert-info">

**Exercice 0.1 -- le toit de l'Europe**

Retrouvez le point le plus haut de la carte.

- Utilisez `Z.argmax()` puis `np.unravel_index(...)` pour obtenir la ligne `i` et la colonne `j` du maximum.
- Affichez son altitude, ainsi que sa latitude `lat[i]` et sa longitude `lon[j]`.
- Vérifiez sur une carte (ou avec vos souvenirs de géographie) de quel sommet il s'agit.

</div>

## 1. La dérivée: le taux d'accroissement

Rappel du cours: la dérivée de $f$ au point $x$ est la limite du **taux d'accroissement**

$$
f'(x) = \lim_{h \rightarrow 0} \frac{f(x+h) - f(x)}{h}.
$$

Un ordinateur ne sait pas calculer une limite. Il ne sait faire qu'une chose: prendre un $h$ **petit mais non nul**, et calculer le quotient. C'est ce qu'on appelle une **différence finie**.

Commençons par une fonction que vous connaissez bien, celle du cours:
$$
f(x) = x^3 - 3x, \qquad f'(x) = 3x^2 - 3.
$$
Comme nous connaissons ici la dérivée exacte, nous allons pouvoir **mesurer l'erreur** commise par l'ordinateur.

In [ ]:
def f(x):
    return x**3 - 3*x

def f_prime_exacte(x):
    return 3*x**2 - 3

<div class="alert alert-block alert-info">

**Exercice 1.1 -- coder la définition de la dérivée**

Écrivez une fonction `derivee(f, x, h)` qui renvoie le taux d'accroissement
$\dfrac{f(x+h) - f(x)}{h}$.

Testez-la en $x = 2$ avec $h = 10^{-6}$, et comparez avec `f_prime_exacte(2)` (qui vaut $9$).

</div>

<div class="alert alert-block alert-info">

**Exercice 1.2 -- quel $h$ faut-il choisir?**

En maths, plus $h$ est petit, meilleure est l'approximation. Vérifions-le sur machine.

- Pour $h = 10^{-1}, 10^{-2}, \dots, 10^{-16}$, calculez l'erreur $|\,\texttt{derivee}(f, 2, h) - f'(2)\,|$.
- Tracez cette erreur en fonction de $h$ avec `plt.loglog` (échelle logarithmique sur les deux axes).
- **Que se passe-t-il?** Le résultat est-il celui que vous attendiez?

</div>

<div class="alert alert-block alert-warning">

**Question.** Décrivez la courbe obtenue. Quel est le $h$ optimal? Pourquoi l'erreur **remonte-t-elle** quand $h$ devient très petit?

</div>

**Réponse.**

*(à compléter)*

### Un profil d'altitude réel

Passons aux vraies données. Nous prenons une **coupe verticale** de la carte: la colonne qui passe par le sommet du Mont Blanc. On obtient une fonction d'une seule variable
$$
p(y) = \text{altitude le long de cette ligne},
$$
c'est-à-dire exactement le **profil** que l'on voit sur les topos de randonnée.

In [ ]:
j_coupe = int(np.argmin(abs(lon - 6.8652)))    # colonne du Mont Blanc
profil  = Z[:, j_coupe]                        # altitudes le long de cette colonne
y       = np.arange(len(profil)) * dy          # distance en mètres, du sud vers le nord

plt.plot(y / 1000, profil)
plt.xlabel("distance vers le nord (km)")
plt.ylabel("altitude (m)")
plt.title("Profil d'altitude passant par le Mont Blanc")
plt.grid(alpha=0.3)
plt.show()

<div class="alert alert-block alert-info">

**Exercice 1.3 -- la pente en pourcentage**

Ici, pas de formule: nous n'avons qu'un tableau de valeurs. La dérivée se calcule donc par différences finies:
$$
p'(y_i) \approx \frac{p(y_{i+1}) - p(y_{i-1})}{2 \, \texttt{dy}}
\qquad \text{(différence \emph{centrée})}.
$$

- Calculez cette dérivée avec `np.gradient(profil, dy)` (qui fait exactement ce calcul pour vous).
- Tracez-la sous le profil.
- La **pente en pourcentage** d'une route est $100 \times |p'|$. Calculez la pente maximale du profil et comparez-la à un panneau routier (un col routier dépasse rarement $12\%$).

</div>

## 2. Le gradient

Une carte d'altitude est une fonction de **deux** variables. Sa dérivée est donc un **vecteur**, le gradient:

$$
\nabla f(x,y) = \left( \begin{matrix} \dfrac{\partial f}{\partial x}(x,y) \\[2mm] \dfrac{\partial f}{\partial y}(x,y) \end{matrix} \right).
$$

`numpy` sait le calculer sur une grille avec `np.gradient(Z, dy, dx)`. Attention à l'ordre: le premier axe d'un tableau, c'est les **lignes** (donc $y$), le second les **colonnes** (donc $x$). La fonction renvoie donc les deux dérivées **dans cet ordre**.

<div class="alert alert-block alert-info">

**Exercice 2.1 -- carte des pentes**

- Calculez `gy, gx = np.gradient(Z, dy, dx)`.
- Calculez la norme du gradient $\|\nabla f\| = \sqrt{g_x^2 + g_y^2}$ (`np.hypot` le fait directement).
- Affichez-la avec `plt.imshow`. Où le terrain est-il le plus raide? Reconnaissez-vous des structures?

</div>

<div class="alert alert-block alert-info">

**Exercice 2.2 -- le théorème du cours, en image**

Le cours affirme que le gradient pointe dans la **direction de plus grande pente ascendante**. Une conséquence visuelle: le gradient est toujours **perpendiculaire aux courbes de niveau**.

Vérifions-le sur un zoom autour du sommet.

- Tracez les courbes de niveau du zoom avec `plt.contour`.
- Superposez les vecteurs gradient avec `plt.quiver` (n'affichez qu'un point sur 8, sinon c'est illisible).
- Les flèches sont-elles perpendiculaires aux courbes? Pointent-elles vers le haut ou vers le bas?

</div>

In [ ]:
# fenêtre de zoom autour du sommet du Mont Blanc
i0, i1, j0, j1 = 60, 180, 175, 295
s = 8      # on n'affiche qu'un vecteur sur 8

Zz  = Z[i0:i1, j0:j1]
gxz = gx[i0:i1, j0:j1]
gyz = gy[i0:i1, j0:j1]

xz = np.arange(j0, j1) * dx / 1000
yz = np.arange(i0, i1) * dy / 1000
XZ, YZ = np.meshgrid(xz, yz)

**Réponse.**

*(à compléter)*

<div class="alert alert-block alert-info">

**Exercice 2.3 -- l'exposition des versants**

La **direction** du gradient (et pas seulement sa norme) a un sens concret: elle indique vers où le terrain monte. Un versant dont le gradient pointe vers le sud est un versant... exposé au nord (à l'ombre), et inversement.

- Calculez l'angle du gradient avec `np.arctan2(gy, gx)`.
- Affichez-le avec une palette circulaire (`cmap="twilight"`).
- Construisez un masque des **versants exposés au sud** (ceux pour lesquels $g_y > 0$, c'est-à-dire qui montent vers le nord) et affichez-le. Ce sont les versants ensoleillés, ceux où l'on installe les panneaux solaires et où la neige fond en premier.

</div>

## 3. Les points critiques: sommets, cuvettes et cols

D'après le théorème de Fermat vu en cours, en un **extremum local** le gradient s'annule:
$$
\nabla f(\vec{x}_0) = \vec{0}.
$$

Sur un terrain, les points où le gradient s'annule sont les endroits **plats**, et ils sont de trois sortes:

- un **maximum local**: un **sommet**;
- un **minimum local**: une **cuvette** (un lac, un fond de vallée fermé);
- un **point selle**: un **col** -- ça monte dans une direction et ça descend dans l'autre. C'est exactement ce qu'est un col de montagne!

<div class="alert alert-block alert-info">

**Exercice 3.1 -- où le gradient s'annule-t-il?**

- Construisez le masque des points où $\|\nabla f\| < 0{,}05$ et affichez-le par-dessus la carte.
- Combien de points obtenez-vous? Sont-ils tous des sommets ou des cols? Que représentent la plupart d'entre eux?

</div>

<div class="alert alert-block alert-info">

**Exercice 3.2 -- retrouver les sommets du massif**

Un sommet, ce n'est pas seulement un point plat: c'est un point **plus haut que tous ses voisins**.

- Lissez légèrement la carte: `Zs = gaussian_filter(Z, 2)` (pour éviter le bruit de mesure).
- Un point est un maximum local si `Zs == maximum_filter(Zs, size=41)`, c'est-à-dire s'il est le plus haut de son voisinage de 41x41 pixels (environ 1 km).
- Ne gardez que ceux au-dessus de 3500 m, affichez-les sur la carte, et listez les 6 plus hauts avec leurs coordonnées.
- Comparez aux altitudes officielles ci-dessous.

</div>

**Réponse.**

*(à compléter)*

<div class="alert alert-block alert-info">

**Exercice 3.3 (bonus) -- distinguer un sommet, une cuvette et un col**

Pour classer un point critique, on regarde les **dérivées secondes**. On calcule
$$
D = \frac{\partial^2 f}{\partial x^2} \cdot \frac{\partial^2 f}{\partial y^2} - \left( \frac{\partial^2 f}{\partial x \, \partial y} \right)^{\!2}
$$
et la règle est la suivante:

| | |
|---|---|
| $D > 0$ et $\partial^2 f / \partial x^2 < 0$ | **maximum local** (sommet) |
| $D > 0$ et $\partial^2 f / \partial x^2 > 0$ | **minimum local** (cuvette) |
| $D < 0$ | **point selle** (col) |

Appliquez cette recette aux points où $\|\nabla f\| < 0{,}03$, et comptez combien il y a de sommets, de cuvettes et de cols.

</div>

## 4. Ruissellement: la descente de gradient

Une goutte d'eau posée sur le terrain suit la ligne de plus grande pente **descendante**, c'est-à-dire la direction $-\nabla f$. C'est exactement l'algorithme de **descente de gradient**:

$$
\vec{x}_{k+1} = \vec{x}_k - \eta \, \nabla f(\vec{x}_k)
$$

où $\eta > 0$ est le **pas** (en machine learning, on l'appelle le *taux d'apprentissage*).

<div class="alert alert-block alert-info">

**Exercice 4.1 -- faire tomber une goutte**

Écrivez une fonction `goutte(x0, y0, eta, n)` qui:

- part du point $(x_0, y_0)$ **exprimé en mètres**;
- répète `n` fois: convertit la position en indices (`j = round(x/dx)`, `i = round(y/dy)`), lit le gradient `gx[i, j]` et `gy[i, j]`, puis applique $\vec{x} \leftarrow \vec{x} - \eta \nabla f$;
- garde la position dans la carte avec `np.clip`;
- renvoie le tableau des positions successives.

Testez-la depuis le point $(3000, 8000)$ avec $\eta = 60$ et $n = 300$, et affichez l'altitude de départ et d'arrivée.

</div>

<div class="alert alert-block alert-info">

**Exercice 4.2 -- une averse**

Lâchez une dizaine de gouttes depuis des points de départ différents et tracez toutes leurs trajectoires par-dessus la carte.

- Toutes les gouttes arrivent-elles au même endroit?
- Que représente le point d'arrivée d'une goutte, en termes du cours?

</div>

**Réponse.**

*(à compléter)*

<div class="alert alert-block alert-info">

**Exercice 4.3 -- l'influence du pas $\eta$**

Reprenez le même point de départ $(3000, 8000)$ et comparez les trajectoires pour $\eta = 5$, $\eta = 60$, $\eta = 3000$ et $\eta = 8000$.

Pour chacune, affichez l'altitude d'arrivée et la longueur moyenne des 50 derniers pas. Que se passe-t-il quand $\eta$ est trop petit? Trop grand?

</div>

**Réponse.**

*(à compléter)*

## Conclusion

Sur une simple carte d'altitude, vous avez manipulé:

- la **dérivée** comme taux d'accroissement, et ses limites en calcul numérique (la courbe en V);
- les **dérivées partielles** et le **gradient** d'une fonction de deux variables;
- le fait que le gradient est **perpendiculaire aux courbes de niveau** et pointe vers le haut;
- les **points critiques** ($\nabla f = \vec{0}$): sommets, cuvettes et cols, et le fait que $\nabla f = \vec 0$ est nécessaire mais pas suffisant;
- la **descente de gradient**, ses minima locaux et le réglage du pas.

**Pour aller plus loin.** Dans la suite du cours, la fonction $f$ ne sera plus une altitude mais une **fonction de coût** $L(\vec{w})$ qui mesure l'erreur d'un modèle, et les coordonnées $\vec{w}$ ne seront plus une position sur une carte mais les **paramètres** du modèle. L'algorithme, lui, sera rigoureusement le même que celui de l'exercice 4.1.